# GramSathi - Model Training Pipeline

Complete walkthrough of training, hyperparameter tuning, interpretation, and export.

**Steps:**
1. Generate/prepare data
2. Preprocess features
3. Train baseline models
4. Hyperparameter tuning with GridSearchCV
5. Model interpretation with SHAP
6. Feature importance visualization
7. Error analysis
8. Model export

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully')

In [ ]:
import sys
sys.path.append('..')
from trainer.data_pipeline import (
    generate_training_dataset, build_preprocessing_pipeline,
    load_from_dataframe, TARGET_COLUMNS, FEATURES
)

MODELS_DIR = "C:\\Users\\User\\gramsathi-ai\\ml\\eligibility-model\\models"
os.makedirs(MODELS_DIR, exist_ok=True)

## 1. Data Preparation

In [ ]:
df = generate_training_dataset(n_samples=100000, seed=42)
X_train, X_test, y_train, y_test = load_from_dataframe(df)
print(f'Training samples: {len(X_train):,}')
print(f'Test samples: {len(X_test):,}')
print(f'Features: {list(X_train.columns)}')
print(f'Target schemes: {TARGET_COLUMNS}')

In [ ]:
preprocessor = build_preprocessing_pipeline()
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

feature_names = []
for name, transformer, columns in preprocessor.transformers_:
    if name == 'num':
        feature_names.extend(columns)
    elif name == 'cat':
        ohe = transformer.named_steps['onehot']
        if hasattr(ohe, 'get_feature_names_out'):
            feature_names.extend(ohe.get_feature_names_out(columns))
        else:
            feature_names.extend([f'{c}_{i}' for c in columns for i in range(len(X_train[c].unique()))])
    elif name == 'bin':
        feature_names.extend(columns)

print(f'Processed feature count: {len(feature_names)}')
print(f'Processed training shape: {X_train_processed.shape}')

## 2. Baseline Model Training

In [ ]:
models = {
    'GradientBoosting': GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'LogisticRegression': LogisticRegression(C=1.0, max_iter=1000, random_state=42, n_jobs=-1)
}

trained_models = {}
results = []

for name, base_model in models.items():
    print(f'\nTraining {name}...')
    multi_model = MultiOutputClassifier(base_model, n_jobs=-1)
    multi_model.fit(X_train_processed, y_train)
    trained_models[name] = multi_model

    y_pred = multi_model.predict(X_test_processed)
    
    for i, scheme in enumerate(TARGET_COLUMNS):
        results.append({
            'Model': name,
            'Scheme': scheme,
            'F1': f1_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
            'Accuracy': accuracy_score(y_test.iloc[:, i], y_pred[:, i]),
            'Precision': precision_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0),
            'Recall': recall_score(y_test.iloc[:, i], y_pred[:, i], zero_division=0)
        })
    
    avg_f1 = np.mean([r['F1'] for r in results if r['Model'] == name])
    print(f'  Avg F1: {avg_f1:.4f}')

results_df = pd.DataFrame(results)
results_df

## 3. Model Comparison Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot_f1 = results_df.pivot(index='Scheme', columns='Model', values='F1')
pivot_f1.plot(kind='bar', ax=axes[0], colormap='Set2')
axes[0].set_title('F1 Score by Model and Scheme', fontsize=12)
axes[0].set_ylabel('F1 Score')
axes[0].legend(loc='lower right')
axes[0].tick_params(axis='x', rotation=45)

pivot_acc = results_df.pivot(index='Scheme', columns='Model', values='Accuracy')
pivot_acc.plot(kind='bar', ax=axes[1], colormap='Set2')
axes[1].set_title('Accuracy by Model and Scheme', fontsize=12)
axes[1].set_ylabel('Accuracy')
axes[1].legend(loc='lower right')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

gb = GradientBoostingClassifier(random_state=42)

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'min_samples_leaf': [10, 20, 50]
}

grid_search = GridSearchCV(
    gb, param_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

print('Running GridSearchCV on first scheme (education_aid)...')
grid_search.fit(X_train_processed, y_train['education_aid'])

print(f'\nBest parameters: {grid_search.best_params_}')
print(f'Best CV score: {grid_search.best_score_:.4f}')

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)

pivot_results = cv_results.pivot_table(
    index='param_max_depth',
    columns='param_n_estimators',
    values='mean_test_score',
    aggfunc='mean'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot_results, annot=True, cmap='YlOrRd', fmt='.4f')
plt.title('Grid Search Results: Max Depth vs N Estimators')
plt.xlabel('N Estimators')
plt.ylabel('Max Depth')
plt.show()

## 5. Train Optimized Multi-Output Model

In [ ]:
best_params = grid_search.best_params_
print(f'Training multi-output model with: {best_params}')

optimized_gb = GradientBoostingClassifier(**best_params, random_state=42)
optimized_model = MultiOutputClassifier(optimized_gb, n_jobs=-1)
optimized_model.fit(X_train_processed, y_train)

y_pred_opt = optimized_model.predict(X_test_processed)

scheme_results = {}
for i, scheme in enumerate(TARGET_COLUMNS):
    y_true, y_pred = y_test.iloc[:, i], y_pred_opt[:, i]
    scheme_results[scheme] = {
        'accuracy': accuracy_score(y_true, y_pred),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0)
    }

opt_results_df = pd.DataFrame(scheme_results).T
opt_results_df['improvement'] = opt_results_df['f1'] - results_df[results_df['Model'] == 'GradientBoosting'].set_index('Scheme').loc[opt_results_df.index, 'F1']
opt_results_df

## 6. Feature Importance Analysis

In [ ]:
importance_data = {}
n_top = 15

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, (ax, scheme) in enumerate(zip(axes, TARGET_COLUMNS)):
    estimator = optimized_model.estimators_[i]
    importances = estimator.feature_importances_
    
    sorted_idx = np.argsort(importances)[::-1][:n_top]
    top_features = [feature_names[j] for j in sorted_idx]
    top_importances = importances[sorted_idx]
    
    importance_data[scheme] = {top_features[j]: float(top_importances[j]) for j in range(n_top)}
    
    colors = plt.cm.Blues(np.linspace(0.4, 1, n_top))
    ax.barh(range(n_top), top_importances[::-1], color=colors[::-1])
    ax.set_yticks(range(n_top))
    ax.set_yticklabels([f[:25] for f in top_features[::-1]], fontsize=8)
    ax.set_title(f"{scheme.replace('_', ' ').title()}", fontsize=10)
    ax.set_xlabel('Importance')

plt.suptitle('Top 15 Feature Importances per Scheme Category', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
consistent_features = set.intersection(*[set(d.keys()) for d in importance_data.values()])
print('Consistently important features across all schemes:')
for f in consistent_features:
    avg_imp = np.mean([importance_data[s][f] for s in TARGET_COLUMNS])
    print(f'  {f}: {avg_imp:.4f}')

## 7. Model Interpretation with SHAP

In [ ]:
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print('SHAP not installed. Install with: pip install shap==0.44.1')

if SHAP_AVAILABLE:
    sample_size = min(200, X_test_processed.shape[0])
    X_sample = X_test_processed[:sample_size]
    
    scheme_idx = 0
    explainer = shap.TreeExplainer(optimized_model.estimators_[scheme_idx])
    shap_values = explainer.shap_values(X_sample)
    
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      title=f"SHAP Summary - {TARGET_COLUMNS[scheme_idx].replace('_', ' ').title()}",
                      show=False)
    plt.tight_layout()
    plt.show()

In [ ]:
if SHAP_AVAILABLE:
    shap.dependence_plot(
        feature_names.index('annual_income'), shap_values[:, :, 1],
        X_sample, feature_names=feature_names,
        show=False, interaction_index='auto'
    )
    plt.title(f"SHAP Dependence: Annual Income ({TARGET_COLUMNS[scheme_idx].replace('_', ' ').title()})")
    plt.show()

## 8. Error Analysis

In [ ]:
X_test_orig = X_test.reset_index(drop=True)
y_test_orig = y_test.reset_index(drop=True)
errors = pd.DataFrame()

for i, scheme in enumerate(TARGET_COLUMNS):
    y_true = y_test_orig.iloc[:, i]
    y_pred = pd.Series(y_pred_opt[:, i])
    
    false_positives = (y_true == 0) & (y_pred == 1)
    false_negatives = (y_true == 1) & (y_pred == 0)
    
    print(f'\n=== {scheme.upper()} ===')
    print(f'  False Positives: {false_positives.sum()}')
    print(f'  False Negatives: {false_negatives.sum()}')
    
    if false_positives.sum() > 0:
        fp_caste = X_test_orig.loc[false_positives.values, 'caste_category'].value_counts()
        print(f'  FP by caste: {dict(fp_caste)}')
        fp_income = X_test_orig.loc[false_positives.values, 'annual_income'].describe()
        print(f'  FP income stats: mean={fp_income["mean"]:.0f}, std={fp_income["std"]:.0f}')
    
    if false_negatives.sum() > 0:
        fn_income = X_test_orig.loc[false_negatives.values, 'annual_income'].describe()
        print(f'  FN income stats: mean={fn_income["mean"]:.0f}, std={fn_income["std"]:.0f}')

## 9. Model Export

In [ ]:
import joblib

model_path = os.path.join(MODELS_DIR, 'eligibility_model.pkl')
preprocessor_path = os.path.join(MODELS_DIR, 'preprocessor.pkl')

joblib.dump(optimized_model, model_path)
joblib.dump(preprocessor, preprocessor_path)
print(f'Model saved to {model_path}')
print(f'Preprocessor saved to {preprocessor_path}')

artifact_manifest = {
    'model_type': 'GradientBoosting',
    'best_params': best_params,
    'features': FEATURES,
    'target_columns': TARGET_COLUMNS,
    'feature_names': feature_names,
    'train_samples': int(len(X_train)),
    'test_samples': int(len(X_test)),
    'per_scheme_metrics': scheme_results
}

with open(os.path.join(MODELS_DIR, 'manifest.json'), 'w') as f:
    json.dump(artifact_manifest, f, indent=2)

print('Manifest saved.')

In [ ]:
print('\n=== Training Pipeline Complete ===')
print(f'Models compared: GradientBoosting, RandomForest, LogisticRegression')
print(f'Best model: GradientBoosting with params {best_params}')
print(f'Average F1 across schemes: {np.mean([v["f1"] for v in scheme_results.values()]):.4f}')
print(f'Artifacts in: {MODELS_DIR}')